this notebook contains the training of the tokeniser, the tokenisation of the data set and then creating neat byte array. some crucial decisions to be made are

1. how large should the vocabulary be?
2. how long should the sequence length be? (context window)
3. how do appropiately chunk and load test data setes? (pytorch loader)

question 2 is particularly relevant as we are most likely to use absolute positional embeddings, which have fixed context window lengths, unlike some of the more modern approaches to positional embeddings. question 1 is also particularly important, and part-art/part-science. a more systematic way to answer the question of vocabulary size would be to empirically test the performance of a fixed network over a range of vocabulary sizes. 

i do not have the compute to do this! since GPT-2 was open sourced, we know that GPT-2 used a vocabulary size of `50,257` (50,256 + an end of sequence token). we can easily affirm that GPT-2 used a much, much larger corpus of training data than just wikipedia pages. thus a vocabulary of ~16k-32k seems appropiate for the knowledge that our network would have to represent. we aim to use a randomly sampled 10% of the training data set to train the tokeniser. we borrow the GPT-2 tokeniser's configs. a larger vocab size means we can generate more complete words, and thus have less tokens per sequence.

before we do anything, we need some data to train the tokeniser. we pull a random sample of 20% of the training dataset for the tokeniser training run.

In [54]:
# get a random sample of training data set data
import duckdb
import os

from huggingface_hub import login, HfApi
from dotenv import load_dotenv

load_dotenv()
path_to_data = os.environ['DATAPATH']
hugging_face_access_token = os.environ['HUGGING_FACE_ACCESS_TOKEN']

login(token=hugging_face_access_token)

con = duckdb.connect()
con.execute("SET memory_limit='24GB'")

In [ ]:
from transformers import AutoTokenizer

# sample 20% of the dataset
cur: duckdb.DuckDBPyConnection = con.execute(
    f"""
    select s.text from '{path_to_data}/train.parquet' s where hash(s.id) % 100 < 20
    """
)

# iterator for batches of text data
def iterate(cur: duckdb.DuckDBPyConnection, batch_size=32):
    while rows := cur.fetchmany(size=batch_size):
        # sql returns tuples within lists
        yield [r[0] for r in rows]

# pull in the gpt2 tokeniser
gpt2_tokenizer = AutoTokenizer.from_pretrained('gpt2')

# train new tokeniser
wiki_tokenizer = gpt2_tokenizer.train_new_from_iterator(iterate(cur), vocab_size=20_000)
wiki_tokenizer.save_pretrained('')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [ ]:
# push to hugging face
wiki_tokenizer.save_pretrained('C:/temp/tokenizer')
HfApi().upload_folder(folder_path='C:/temp/tokenizer', repo_id='notoverfit/askwiki', path_in_repo='tokenizer')

CommitInfo(commit_url='https://huggingface.co/notoverfit/askwiki/commit/20ee94cc296d03e8e767e11202480408cd486721', commit_message='Upload folder using huggingface_hub', commit_description='', oid='20ee94cc296d03e8e767e11202480408cd486721', pr_url=None, repo_url=RepoUrl('https://huggingface.co/notoverfit/askwiki', endpoint='https://huggingface.co', repo_type='model', repo_id='notoverfit/askwiki'), pr_revision=None, pr_num=None)

before tokenising the dataset, first consider an appropiate context window and embedding dimension. given we look to use absolute positional embedding, the choice of the context window is important since there is a fixed number of tokens that the model can attend to. 

furthermore, context window is an input into the computation requirement of the network. since the smallest GPT-2 model (~124M parameters) is of similar size to my intended size (maxing out at ~250M); choose the same context window of 1,024 tokens.

In [49]:
import numpy as np
import pyarrow.parquet as pq

eos = wiki_tokenizer.eos_token_id

def write_bin(split):
    pf = pq.ParquetFile(f'{path_to_data}/{split}.parquet')
    n_tokens = 0
    with open(f'{path_to_data}/{split}.bin', 'wb') as f:
        for batch in pf.iter_batches(batch_size=10_000, columns=['text']):
            texts = [t for t in batch.column('text').to_pylist() if t]
            ids = wiki_tokenizer(texts)['input_ids']
            flat = np.concatenate([np.array(x + [eos], dtype=np.uint16) for x in ids])
            flat.tofile(f)
            n_tokens += len(flat)
    print(f'{split}: {n_tokens:,} tokens')

for split in ['train', 'val', 'test']:
    write_bin(split)

train: 4,274,841,294 tokens
val: 13,301,747 tokens
test: 13,395,179 tokens


according to the Chinchilla scaling laws, there should be roughly a 20:1 ratio of tokens per parameter. thus with our training set size, we could theoretically train a model of ~213M parameters; which is close to the upper bound of our intended size (250M). given this, the actual trained model will likely be ~200M parameters.

we now can develop a data loader class, which chunks the bytestream into relevant batches of the desired sequence length.

In [85]:
import os
import numpy as np
import torch
from torch.utils.data import Dataset

class TokenSequence(Dataset):
    def __init__(self, path: str, dtype: type, seq_len: int):
        if not os.path.exists(path) or not path.endswith('.bin'):
            raise FileNotFoundError('TokenSequence: path does not exist or is not a binary (.bin) file.')

        self.data = np.memmap(path, dtype=dtype, mode="r")
        self.seq_len = seq_len
        self.dtype = dtype

    def __len__(self):
        """
        returns the length of the data set, measured by number of sequences.
        if the data set is not perfectly divisible by the sequence length, 
        the last partial sequence is deleted. remove one token for the target.
        """
        return (self.data.shape[0] - 1) // self.seq_len

    def __getitem__(self, idx: int):
        if idx > self.__len__() - 1 or idx < 0:
            raise IndexError('TokenSequence: attempt to get index out of range.')

        # get sequence length + 1 of tokens, for shifted target tensor
        start_idx = idx * self.seq_len
        chunk = self.data[start_idx:start_idx+self.seq_len+1].astype(self.dtype)

        # cast to tensors
        x = torch.from_numpy(chunk[:-1])
        y = torch.from_numpy(chunk[1:])

        return x, y

to validate the above, test getting a single training example. the second tensor should be a shifted version of the first, representing the predictor and target tensors. now, we are ready to conduct pretraining.

In [ ]:
x = TokenSequence('C:/data/wikipedia/train.bin', np.uint16, 1_024)
x.__getitem__(0)

(tensor([  427,  2969, 10362,  ...,    14,   370,   398], dtype=torch.uint16),
 tensor([ 2969, 10362,   314,  ...,   370,   398,  2609], dtype=torch.uint16))

: 